# 02 — Panel de hogares (pares t→t+1 y t→t+4)

Vincula hogares entre un trimestre de origen `t` y `t+h` (`h=1` o `h=4`) sobre
`data/02_nucleo/` -- **no construye ninguna etiqueta de deterioro**, solo el panel
vinculado con sus flags. Resuelve el pedido de la directora de justificar cuántos
pares hay por horizonte con datos medidos, y deja documentado (sin resolver) el
momento de observación dentro del trimestre.

Lógica de vinculación/clasificación en `src/panel.py` (con tests sintéticos en
`tests/test_panel.py`). Sigue las convenciones del resto del pipeline: `TRIMESTRES`/
`REHACER`, `pipeline_meta`, `resumen_03_panel.csv` incremental, `corridas.csv` final.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd

def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

from armonizacion import trimestres_publicados, es_historico, pendiente_rar
import tipado_nucleo as tn
import panel
import pipeline_meta as pm

TRIMESTRES: list[tuple[int, int]] | None = None  # subconjunto opcional (trimestres de ORIGEN t); None = todos
REHACER: bool = False  # True fuerza reprocesar aunque la partición esté al día
HORIZONTES = (1, 4)

NUCLEO = REPO / "data" / "02_nucleo"
PANEL = REPO / "data" / "03_panel"
META = REPO / "data" / "meta"
for _h in HORIZONTES:
    (PANEL / f"pares_h{_h}").mkdir(parents=True, exist_ok=True)

trimestres_origen = TRIMESTRES if TRIMESTRES is not None else trimestres_publicados()
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
COMMIT_GIT = pm.obtener_commit_git(REPO)
print(f"{len(trimestres_origen)} trimestres de origen a procesar" +
      (" -- CORRIDA PARCIAL (TRIMESTRES acotado)" if ES_CORRIDA_PARCIAL else ""))

**Advertencia de esquema** (no bloquea): compara el hash de `ESQUEMA_NUCLEO` vigente
contra el que se usó para tipar cada partición de `02_nucleo` (`resumen_02_nucleo.csv`,
columna `esquema_hash`). `particion_desactualizada()` solo compara mtimes de datos --
esto cubre el caso de un cambio de código en `tipado_nucleo.py` que no se reflejó
todavía en una nueva corrida de 01b con `REHACER=True`.

In [ ]:
_resumen_02 = META / "resumen_02_nucleo.csv"
if _resumen_02.exists():
    _r = pd.read_csv(_resumen_02)
    _hash_actual = tn.hash_esquema()
    _desactualizadas = _r[_r["esquema_hash"] != _hash_actual] if "esquema_hash" in _r.columns else _r
    if len(_desactualizadas):
        print(f"ADVERTENCIA: {len(_desactualizadas)} particiones de 02_nucleo fueron tipadas "
              f"con un ESQUEMA_NUCLEO distinto al vigente ({_hash_actual}) -- puede faltarles "
              f"una variable agregada después. Trimestres afectados:")
        display(_desactualizadas[["anio", "trimestre", "tipo", "esquema_hash"]].drop_duplicates())
    else:
        print(f"OK: las {len(_r)} particiones de 02_nucleo están tipadas con el esquema vigente ({_hash_actual}).")
else:
    print("resumen_02_nucleo.csv no existe todavía -- correr notebook 01b primero.")

## Paso 0 — pendientes previos (verificados en Plan Mode, antes de esta corrida)

1. **`P21` nulo (NaN) entre ocupados**: 0% en los 81 trimestres publicados (pre y post
   2016) -- no hay NaN en magnitud comparable al 15.8% de `P21=-9` post-2016. **D3 no
   necesita revisión de fondo**; se propuso (no escrita todavía) una línea adicional
   para D3 confirmando esto explícitamente.
2. **Assert `DECCFR==12 ⊆ ingreso_no_declarado`**: ya existe en
   `tipado_nucleo.ingreso_no_declarado()` (línea con el `assert`). Se re-ejecuta contra
   los datos reales en la celda siguiente, no solo se cita en prosa.
3. **`CH03` (relación de parentesco)**: faltaba en `ESQUEMA_NUCLEO`, se agregó
   (`base="individual"`, códigos 1-10 confirmados contra el codebook p.16 y contra los
   81/81 trimestres). El notebook 01b se re-corrió una vez con `REHACER=True` para
   incorporarla (cambio de esquema/código, no de datos de origen -- no lo detecta
   `particion_desactualizada`). Se agregó además `tipado_nucleo.hash_esquema()` +
   columna `esquema_hash` en `resumen_02_nucleo.csv`, con el chequeo de la celda
   anterior para no repetir este tipo de desfasaje en silencio la próxima vez.

In [ ]:
_n_chequeados = 0
for _anio, _trimestre in trimestres_origen:
    if es_historico(_anio, _trimestre):
        continue
    _p = pm.ruta_particion(NUCLEO, "hogar", _anio, _trimestre)
    if not _p.exists():
        continue
    _hogar = pd.read_parquet(_p, columns=["CODUSU", "NRO_HOGAR", "DECCFR", "PONDIH"])
    tn.ingreso_no_declarado(_hogar, es_historico=False)  # dispara el assert interno si falla
    _n_chequeados += 1
print(f"assert DECCFR==12 ⊆ ingreso_no_declarado: OK en los {_n_chequeados} trimestres regulares verificados.")

## Paso 4 — momento de observación (adelantado)

- **[confirmado]** El codebook define `TRIMESTRE` (p.14) solo como "ventana de
  observación" 1-4 por año. La condición de actividad se refiere a la "semana de
  referencia" (p.20) y los ingresos de la ocupación principal a un "mes de referencia"
  (p.29, Anexo I p.31) -- ambos relativos a la fecha de entrevista de cada caso, no
  fijados por trimestre.
- **[no cubierto]** Este PDF es un "Diseño de registros", no el documento de diseño
  muestral operativo de INDEC -- no dice cómo se distribuye la muestra a lo largo de
  las ~13 semanas del trimestre. Tampoco existe ninguna variable en los microdatos que
  identifique semana/mes de relevamiento del caso (confirmado contra
  `data/meta/inventario_variables.csv`: las únicas variables con "MES"/"SEM" son de
  preguntas de historia laboral puntuales, no la fecha de la entrevista en sí).
- **Conclusión para la directora**: el momento de observación no se puede precisar con
  lo disponible en este repo. Requeriría el documento de diseño muestral de INDEC
  (fuera de este PDF) o asumir el punto medio del trimestre como aproximación
  **[inferido, pendiente de decisión de la directora]**.

## Paso 1 y 2 — vinculación, validación, y escritura del panel

In [ ]:
def ruta_pares(base_dir: Path, anio: int, trimestre: int) -> Path:
    return base_dir / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"


def construir_atricion_origen(hogar_t: pd.DataFrame, individual_t: pd.DataFrame,
                               clasificacion_vivienda: pd.DataFrame, anio_t: int, trimestre_t: int, h: int) -> pd.DataFrame:
    """Tabla a nivel hogar EN t (no pares) para la atrición descriptiva del Paso 3:
    ¿los hogares que terminan vinculados en t+h se parecen a los que no, en t?
    `clasificacion_vivienda` es la salida de `panel.clasificar_viviendas_no_vinculadas`
    (una fila por CODUSU con `estado_vinculacion`: vinculada/hogar_reemplazado/vivienda_sin_par)."""
    df = hogar_t[["CODUSU", "NRO_HOGAR", "IPCF", "AGLOMERADO"]].copy()
    df = df.merge(clasificacion_vivienda, on="CODUSU", how="left")
    n_miembros = individual_t.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros")
    jefe = (individual_t[individual_t["CH03"] == 1]
            .drop_duplicates(["CODUSU", "NRO_HOGAR"])
            .set_index(["CODUSU", "NRO_HOGAR"])["ESTADO"].rename("estado_jefe"))
    df = df.join(n_miembros, on=["CODUSU", "NRO_HOGAR"]).join(jefe, on=["CODUSU", "NRO_HOGAR"])
    df["anio_t"] = anio_t
    df["trimestre_t"] = trimestre_t
    df["h"] = h
    return df

In [ ]:
tasas_viviendas_filas = []
pares_acumulados = {1: [], 4: []}
componentes_acumulados = {1: [], 4: []}
atricion_acumulada = {1: [], 4: []}
resumen_filas_totales = []

t0 = time.time()
n_reusados = 0
n_escritos = 0
for anio_t, trimestre_t in trimestres_origen:
    hogar_t_path = pm.ruta_particion(NUCLEO, "hogar", anio_t, trimestre_t)
    ind_t_path = pm.ruta_particion(NUCLEO, "individual", anio_t, trimestre_t)
    if not hogar_t_path.exists() or not ind_t_path.exists():
        continue  # sin 02_nucleo para este origen (ver resumen_02_nucleo.csv del notebook 01b)

    hogar_t = pd.read_parquet(hogar_t_path)
    individual_t = pd.read_parquet(ind_t_path)
    historico_t = es_historico(anio_t, trimestre_t)
    codusu_t = set(hogar_t["CODUSU"])

    filas_resumen_trimestre = []
    for h in HORIZONTES:
        anio_th, trimestre_th = panel.sumar_trimestres(anio_t, trimestre_t, h)
        hogar_th_path = pm.ruta_particion(NUCLEO, "hogar", anio_th, trimestre_th)
        ind_th_path = pm.ruta_particion(NUCLEO, "individual", anio_th, trimestre_th)
        destino_existe_origen = hogar_th_path.exists() and ind_th_path.exists()

        if not destino_existe_origen:
            tasas_viviendas_filas.append({
                "anio": anio_t, "trimestre": trimestre_t, "h": h, "par_existe": False,
                "n_viviendas_t": len(codusu_t), "n_vinculadas": None, "tasa": None,
                "n_hogar_reemplazado": None, "n_vivienda_sin_par": None,
            })
            filas_resumen_trimestre.append({
                "anio": anio_t, "trimestre": trimestre_t, "h": h, "par_existe": False,
                "n_pares_hogar": 0, "reusado": False,
            })
            continue

        hogar_th = pd.read_parquet(hogar_th_path)
        individual_th = pd.read_parquet(ind_th_path)
        historico_th = es_historico(anio_th, trimestre_th)
        codusu_th = set(hogar_th["CODUSU"])

        tasa = panel.tasa_vinculacion_viviendas(codusu_t, codusu_th)

        panel_dir = PANEL / f"pares_h{h}"
        destino_pares = ruta_pares(panel_dir, anio_t, trimestre_t)
        origenes = [hogar_t_path, ind_t_path, hogar_th_path, ind_th_path]
        desactualizada = REHACER or not destino_pares.exists() or any(pm.particion_desactualizada(o, destino_pares) for o in origenes)

        if not desactualizada:
            pares = pd.read_parquet(destino_pares)
            reusado = True
            n_reusados += 1
        else:
            pares = panel.construir_pares(hogar_t, hogar_th, individual_t, individual_th,
                                           anio_t, trimestre_t, h, historico_t, historico_th)
            destino_pares.parent.mkdir(parents=True, exist_ok=True)
            pares.to_parquet(destino_pares, index=False)
            reusado = False
            n_escritos += 1

        # IPCF y cantidad de miembros de ambas puntas, solo para el resumen final en
        # memoria (D11) -- no se persisten en el Parquet de pares, ya escrito arriba.
        ipcf_t = hogar_t[["CODUSU", "NRO_HOGAR", "IPCF"]].rename(columns={"NRO_HOGAR": "NRO_HOGAR_t", "IPCF": "IPCF_t"})
        ipcf_th = hogar_th[["CODUSU", "NRO_HOGAR", "IPCF"]].rename(columns={"NRO_HOGAR": "NRO_HOGAR_th", "IPCF": "IPCF_th"})
        n_miembros_t = individual_t.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros_t")
        n_miembros_th = individual_th.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros_th")
        pares = pares.merge(ipcf_t, on=["CODUSU", "NRO_HOGAR_t"], how="left").merge(ipcf_th, on=["CODUSU", "NRO_HOGAR_th"], how="left")
        pares = pares.merge(n_miembros_t, left_on=["CODUSU", "NRO_HOGAR_t"], right_index=True, how="left")
        pares = pares.merge(n_miembros_th, left_on=["CODUSU", "NRO_HOGAR_th"], right_index=True, how="left")

        pares_acumulados[h].append(pares)

        clasificacion_vivienda = panel.clasificar_viviendas_no_vinculadas(hogar_t, hogar_th, pares)
        conteo_estados = clasificacion_vivienda["estado_vinculacion"].value_counts()
        tasas_viviendas_filas.append({
            "anio": anio_t, "trimestre": trimestre_t, "h": h, "par_existe": True, **tasa,
            "n_hogar_reemplazado": int(conteo_estados.get("hogar_reemplazado", 0)),
            "n_vivienda_sin_par": int(conteo_estados.get("vivienda_sin_par", 0)),
        })

        componentes = panel.emparejar_componentes(individual_t, individual_th, pares)
        componentes_acumulados[h].append(componentes)

        atricion_acumulada[h].append(construir_atricion_origen(hogar_t, individual_t, clasificacion_vivienda, anio_t, trimestre_t, h))

        filas_resumen_trimestre.append({
            "anio": anio_t, "trimestre": trimestre_t, "h": h, "par_existe": True,
            "n_pares_hogar": len(pares), "reusado": reusado,
        })

    resumen_filas_totales.extend(filas_resumen_trimestre)
    pm.actualizar_resumen_trimestre(META / "resumen_03_panel.csv", filas_resumen_trimestre, claves=("anio", "trimestre", "h"))

duracion_total = time.time() - t0
print(f"tiempo total: {duracion_total:.1f}s")
print(f"pares escritos (nuevo): {n_escritos} -- reusados (al día): {n_reusados}")

### Paso 1 — tasa de vinculación de viviendas

In [ ]:
import matplotlib.pyplot as plt

tasas_viviendas = pd.DataFrame(tasas_viviendas_filas)
tasas_viviendas["periodo"] = tasas_viviendas.apply(lambda r: pd.Period(f"{r['anio']}Q{r['trimestre']}", freq="Q"), axis=1)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for ax, h in zip(axes, HORIZONTES):
    serie = tasas_viviendas[tasas_viviendas["h"] == h].sort_values("periodo")
    ax.plot(serie["periodo"].dt.to_timestamp(), serie["tasa"] * 100, marker=".", label=f"tasa vinculación h={h}")
    ax.axhline(50, linestyle=":", color="gray", label="~50% teórico esquema 2-2-2 [inferido]")
    for anio_hueco, trimestre_hueco in [(2007, 3), (2015, 3), (2015, 4), (2016, 1)]:
        fecha = pd.Period(f"{anio_hueco}Q{trimestre_hueco}", freq="Q").to_timestamp()
        ax.axvline(fecha, color="red", alpha=0.3, linewidth=1)
    for anio_rar, trimestre_rar in [(2014, 1), (2014, 3), (2014, 4), (2015, 1), (2015, 2)]:
        fecha = pd.Period(f"{anio_rar}Q{trimestre_rar}", freq="Q").to_timestamp()
        ax.axvline(fecha, color="orange", alpha=0.3, linewidth=1)
    ax.set_title(f"h={h} (rojo = sin datos publicados, naranja = .rar pendiente)")
    ax.set_ylabel("% viviendas vinculadas")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for h in HORIZONTES:
    serie = tasas_viviendas[(tasas_viviendas["h"] == h) & tasas_viviendas["par_existe"]]
    print(f"h={h}: tasa promedio = {serie['tasa'].mean():.1%} sobre {len(serie)} pares existentes "
          f"(de {len(tasas_viviendas[tasas_viviendas['h']==h])} orígenes)")

**Actualización tras correr con datos reales**: la pregunta de abajo (¿INDEC mantuvo
la muestra a través del hueco?) **no se puede responder con esta comparación** -- se
encontró un motivo técnico previo y más básico. `CODUSU` tiene un **formato
completamente distinto entre la era histórica y la regular**: en la era histórica
(DBF, 2003T3-2015T4) es un código puramente numérico de 6 dígitos (ej. `"393389"`,
verificado en 2010T1/2013T4/2014T2); en la era regular (texto, 2016T1 en adelante) es
un código alfanumérico de 29 caracteres (ej. `"TQRMNOQUXHKOKMCDEGKDB00475140"`). Son
esquemas de identificación incompatibles, no la misma clave en dos formatos --
**cualquier par que cruce la frontera histórico/regular tiene 0% de vinculación por
construcción**, sin importar si INDEC mantuvo la muestra real o no. Confirmado
empíricamente: la intersección de `CODUSU` entre 2015T2 (histórico) y 2016T2
(regular) es exactamente 0, mientras que 2016T2 vs. 2016T3 (ambos regulares) comparten
8,378 códigos.

En el panel actual esto afecta a un solo par: `2015T2→2016T2` (h=4) es el único origen
histórico cuyo destino h=4 cae en la era regular y además existe (2015T1→2016T1 y
2015T3/T4→2016T2/T3 no existen por el hueco de publicación 2015T3-2016T1). No es una
limitación general del panel, es específica de este único par -- verificado en la
tabla de abajo (`tasa=0.0` exactamente en 2015T2, vs. ~43-44% en sus vecinos).

**No documentado hasta ahora** en `CLAUDE.md` ni en `docs/decisiones_metodologicas.md`
-- propuesto como entrada nueva (D7, no escrita todavía): "`CODUSU` cambia de formato
entre la era histórica (numérico, 6 dígitos) y la regular (alfanumérico, 29
caracteres) -- ningún par de panel puede cruzar esa frontera por `CODUSU`, incluso si
la vivienda real continuó en la muestra." La tabla de abajo queda como verificación
numérica de este hallazgo, no como pregunta abierta.

In [ ]:
vecinos_hueco = [(2014, 2), (2014, 3), (2014, 4), (2015, 1), (2015, 2), (2016, 2), (2016, 3), (2016, 4)]
tabla_vecinos = tasas_viviendas[(tasas_viviendas["h"] == 4) & tasas_viviendas.apply(lambda r: (r["anio"], r["trimestre"]) in vecinos_hueco, axis=1)].copy()
tabla_vecinos["es_rar_pendiente"] = tabla_vecinos.apply(lambda r: pendiente_rar(r["anio"], r["trimestre"]), axis=1)
promedio_general_h4 = tasas_viviendas[(tasas_viviendas["h"] == 4) & tasas_viviendas["par_existe"]]["tasa"].mean()
print(f"promedio general de la serie h=4: {promedio_general_h4:.1%}")
tabla_vecinos[["anio", "trimestre", "par_existe", "tasa", "es_rar_pendiente"]].sort_values(["anio", "trimestre"])

### Paso 2 — vinculación y validación de hogares

In [ ]:
pares_todos = {h: pd.concat(partes, ignore_index=True) if partes else pd.DataFrame() for h, partes in pares_acumulados.items()}
componentes_todos = {h: pd.concat(partes, ignore_index=True) if partes else pd.DataFrame() for h, partes in componentes_acumulados.items()}

print("--- diagnóstico previo: ¿COMPONENTE identifica a la misma persona entre visitas? ---")
for h in HORIZONTES:
    r = panel.tasa_coincidencia_componente(componentes_todos[h])
    print(f"h={h}: {r['n_componentes_matcheados']:,} componentes matcheados, "
          f"tasa de mismo sexo = {r['tasa_mismo_sexo']:.1%}" if r["tasa_mismo_sexo"] is not None else f"h={h}: sin componentes matcheados")

**Viviendas no vinculadas, por qué** (join exacto `CODUSU+NRO_HOGAR+AGLOMERADO`,
ver D9): `hogar_reemplazado` (la vivienda sigue en `t+h` pero ningún hogar
matcheó exacto -- confirmado que >96% de estos casos no tiene composición
coincidente, ver diagnóstico de la tarea) vs. `vivienda_sin_par` (la vivienda
ya no está en `t+h`).

Dos lecturas de `hogar_reemplazado`, según el denominador: sobre **todas**
las viviendas de origen (incluidas las que ni siquiera sobreviven como
vivienda) da un porcentaje chico; sobre las viviendas que **sí** siguen
existiendo en `t+h` (aunque con otro hogar) da el número comparable al
diagnóstico que definió el join exacto -- se muestran ambos abajo.

`vivienda_sin_par` mezcla dos fenómenos: la rotación 2-2-2 del panel (~50%
teórico, ver el gráfico de la sección anterior) y atrición real (la vivienda
debería seguir en el esquema pero no aparece -- mudanza, rechazo, no
localización). El excedente sobre el 50% teórico es la atrición real:

In [ ]:
for h in HORIZONTES:
    serie = tasas_viviendas[(tasas_viviendas["h"] == h) & tasas_viviendas["par_existe"]]
    n_t = serie["n_viviendas_t"].sum()
    n_vinculadas_vivienda = serie["n_vinculadas"].sum()
    n_reemplazado = serie["n_hogar_reemplazado"].sum()
    n_sin_par = serie["n_vivienda_sin_par"].sum()
    pct_sin_par = n_sin_par / n_t * 100
    print(f"h={h}: de {n_t:,} viviendas-origen")
    print(f"  hogar_reemplazado: {n_reemplazado:,} -- {n_reemplazado/n_t:.2%} de TODAS las viviendas de origen, "
          f"{n_reemplazado/n_vinculadas_vivienda:.2%} de las que siguen existiendo como vivienda en t+h")
    print(f"  vivienda_sin_par: {n_sin_par:,} ({pct_sin_par:.2f}% de las viviendas-origen) -- "
          f"~50 pp esperables por la rotación 2-2-2, excedente de {pct_sin_par - 50:.1f} pp atribuible a atrición real")

**Distribuciones antes de fijar ningún corte de tolerancia** (pedido explícito de
la tarea) -- separadas por horizonte, porque el tiempo transcurrido es distinto
(~3 meses para h=1, ~12 meses para h=4):

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for i, h in enumerate(HORIZONTES):
    df = pares_todos[h]
    componentes_h = componentes_todos[h]

    axes[i, 0].hist(df["diferencia_edad_jefe"].dropna(), bins=range(-5, 15))
    axes[i, 0].set_title(f"h={h}: diferencia edad jefe/a")

    axes[i, 1].hist(componentes_h["diferencia_edad"].dropna(), bins=range(-5, 15))
    axes[i, 1].set_title(f"h={h}: diferencia edad miembros (por COMPONENTE)")

    axes[i, 2].hist((df["n_altas"] + df["n_bajas"]).dropna(), bins=range(0, 8))
    axes[i, 2].set_title(f"h={h}: altas+bajas por hogar (sexo+edad, no COMPONENTE)")
plt.tight_layout()
plt.show()

**Identidad del hogar-par por núcleo** (`panel.identidad_nucleo`, D10 regla (a),
implementada): `identidad_confirmada` es `True` si el/la jefe/a de `t` (`CH03==1`)
aparece en el núcleo de `t+h` (`CH03` 1 **o** 2 -- permite el intercambio de rol
jefe/a↔cónyuge entre visitas) con mismo sexo y edad dentro de
`TOLERANCIA_EDAD_POR_HORIZONTE[h]` (provisoria, D6). `NA` si el/la jefe/a de `t`
es ambiguo/a. La composición completa (altas/bajas de integrantes) queda aparte,
en `n_altas`/`n_bajas`/`cambio_composicion` -- no entra en la identidad, mismo
criterio que `cambio_jefatura` (tampoco entra):

In [ ]:
for h in HORIZONTES:
    df = pares_todos[h]
    if df.empty:
        continue
    print(f"--- h={h} ---")
    print(df["identidad_confirmada"].value_counts(normalize=True, dropna=False).round(3))
    print(f"cambio_composicion: {df['cambio_composicion'].mean():.1%}")
    print(f"cambio_jefatura: {df['cambio_jefatura'].mean():.1%}")
    print()

### Verificación: ¿la identidad por núcleo depende del tamaño del hogar? (D10)

La clasificación anterior (proporción de `COMPONENTE` coincidentes, D9) exigía que
coincidiera casi **todo** el hogar para `confirmado` -- eso penalizaba mecánicamente a
los hogares grandes (ver D10: `confirmado` caía de 90% a 62% entre 1 y 6+ miembros, sin
relación con si el hogar era "el mismo"). La regla (a) implementada (`identidad_nucleo`,
celda anterior) se apoya solo en el/la jefe/a -- el `%` de `identidad_confirmada` **no
debería** caer con el tamaño del hogar. Se verifica abajo; si cae, se reporta.

In [ ]:
for h in HORIZONTES:
    atricion_h = pd.concat(atricion_acumulada[h], ignore_index=True)[
        ["CODUSU", "NRO_HOGAR", "anio_t", "trimestre_t", "n_miembros"]
    ].rename(columns={"NRO_HOGAR": "NRO_HOGAR_t", "anio_t": "ANO4_t", "trimestre_t": "TRIMESTRE_t"})
    df = pares_todos[h].merge(atricion_h, on=["CODUSU", "NRO_HOGAR_t", "ANO4_t", "TRIMESTRE_t"], how="left")
    df["tamano"] = df["n_miembros"].clip(upper=6)

    tabla = df.groupby("tamano")["identidad_confirmada"].value_counts(dropna=False).unstack(fill_value=0)
    for col in [True, False]:
        if col not in tabla.columns:
            tabla[col] = 0
    total = tabla.sum(axis=1)
    resumen = pd.DataFrame({
        "n_pares": total,
        "identidad_confirmada_pct": (tabla[True] / total * 100).round(1),
        "identidad_no_confirmada_pct": (tabla[False] / total * 100).round(1),
        "jefe_ambiguo_pct": ((total - tabla[True] - tabla[False]) / total * 100).round(1),
        "cambio_composicion_pct": (df.groupby("tamano")["cambio_composicion"].mean() * 100).round(1),
    })
    resumen.index = [str(i) if i < 6 else "6+" for i in resumen.index]
    print(f"--- h={h}: identidad por núcleo, por tamaño de hogar en t (D10, regla (a)) ---")
    display(resumen)

**Unipersonales, aparte**: en un hogar de 1 miembro el núcleo es esa única persona --
si no coincide, no hay margen para otra interpretación que un cambio de ocupante (la
persona que vivía sola se fue, y quien está ahora es otra distinta):

In [ ]:
for h in HORIZONTES:
    atricion_h = pd.concat(atricion_acumulada[h], ignore_index=True)[
        ["CODUSU", "NRO_HOGAR", "anio_t", "trimestre_t", "n_miembros"]
    ].rename(columns={"NRO_HOGAR": "NRO_HOGAR_t", "anio_t": "ANO4_t", "trimestre_t": "TRIMESTRE_t"})
    df = pares_todos[h].merge(atricion_h, on=["CODUSU", "NRO_HOGAR_t", "ANO4_t", "TRIMESTRE_t"], how="left")
    unipersonales = df[df["n_miembros"] == 1]
    n_total = len(unipersonales)
    n_no_confirmada = int((unipersonales["identidad_confirmada"] == False).sum())  # noqa: E712
    print(f"h={h}: {n_total:,} hogares unipersonales en t -- {n_no_confirmada:,} "
          f"({n_no_confirmada/n_total:.1%}) con identidad NO confirmada (probable cambio de ocupante)")

**¿`COMPONENTE` se reasigna entre visitas?** Para los componentes que no coinciden
(sexo+edad) bajo el mismo número de `COMPONENTE`, se busca si existe otra persona del
mismo hogar en `t+h`, bajo un `COMPONENTE` distinto, que sí coincida en sexo y edad (en
ventana) con la persona de `t`. Si aparece, es evidencia de que INDEC renumeró a esa
persona entre visitas, no de que cambió la composición real del hogar.

In [ ]:
resultados_reasignacion = {1: {"fallidos": 0, "explicados": 0}, 4: {"fallidos": 0, "explicados": 0}}

for h in HORIZONTES:
    minimo, maximo = panel.TOLERANCIA_EDAD_POR_HORIZONTE[h]
    for anio_t, trimestre_t in trimestres_origen:
        pares_path = ruta_pares(PANEL / f"pares_h{h}", anio_t, trimestre_t)
        ind_t_path = pm.ruta_particion(NUCLEO, "individual", anio_t, trimestre_t)
        if not pares_path.exists() or not ind_t_path.exists():
            continue
        pares_hogar = pd.read_parquet(pares_path, columns=["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th"])
        if pares_hogar.empty:
            continue
        anio_th, trimestre_th = panel.sumar_trimestres(anio_t, trimestre_t, h)
        ind_th_path = pm.ruta_particion(NUCLEO, "individual", anio_th, trimestre_th)
        if not ind_th_path.exists():
            continue
        individual_t = pd.read_parquet(ind_t_path, columns=["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06"])
        individual_th = pd.read_parquet(ind_th_path, columns=["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06"])

        emp = panel.emparejar_componentes(individual_t, individual_th, pares_hogar)
        emp["coincide"] = emp["mismo_sexo"] & emp["diferencia_edad"].between(minimo, maximo)
        fallidos = emp[~emp["coincide"]]
        if fallidos.empty:
            continue

        izq = fallidos.merge(
            individual_t[["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06"]],
            left_on=["CODUSU", "NRO_HOGAR_t", "COMPONENTE"], right_on=["CODUSU", "NRO_HOGAR", "COMPONENTE"],
        ).drop(columns=["NRO_HOGAR"])
        candidatos = individual_th[["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06"]].rename(columns={
            "NRO_HOGAR": "NRO_HOGAR_th", "COMPONENTE": "COMPONENTE_cand", "CH04": "CH04_cand", "CH06": "CH06_cand",
        })
        combinado = izq.merge(candidatos, on=["CODUSU", "NRO_HOGAR_th"])
        combinado = combinado[combinado["COMPONENTE_cand"] != combinado["COMPONENTE"]]

        resultados_reasignacion[h]["fallidos"] += len(fallidos)
        if combinado.empty:
            continue
        combinado["diferencia"] = combinado["CH06_cand"].astype("Int64") - combinado["CH06"].astype("Int64")
        combinado["coincide_otro"] = (combinado["CH04_cand"] == combinado["CH04"]) & combinado["diferencia"].between(minimo, maximo)
        explicados = combinado.groupby(["CODUSU", "NRO_HOGAR_t", "COMPONENTE"])["coincide_otro"].any()
        resultados_reasignacion[h]["explicados"] += int(explicados.sum())

for h in HORIZONTES:
    r = resultados_reasignacion[h]
    pct = r["explicados"] / r["fallidos"] * 100 if r["fallidos"] else 0
    print(f"h={h}: {r['fallidos']:,} componentes no coinciden bajo el mismo número -- "
          f"{r['explicados']:,} ({pct:.1f}%) tienen un candidato de sexo+edad compatible bajo otro COMPONENTE del mismo hogar")

**Regla (a) implementada (D10)**: identidad por núcleo (`identidad_nucleo`) en vez de
proporción de `COMPONENTE` coincidentes. La composición completa queda como flag aparte
(`n_altas`/`n_bajas`/`cambio_composicion`), sin entrar en la identidad -- mismo criterio
que `cambio_jefatura`.

**Corrección sobre la reasignación de `COMPONENTE`** (el ~7% de arriba): esa cifra es
un **piso**, no una partición completa de "por qué no coincide". Los desajustes que la
reasignación de `COMPONENTE` no explica mezclan dos fenómenos que estos datos no
permiten separar: cambio real de integrantes (alguien se fue, alguien nuevo llegó) y
error de declaración de edad (la misma persona, pero `CH06` corrido más allá de la
ventana de tolerancia por redondeo o error de carga). No hay una variable en la EPH que
distinga ambos casos -- se documenta como límite del método, no se fuerza una
atribución.

## `cambio_composicion` vs. cambio en la cantidad de miembros (D11)

`cambio_composicion` (D10) da 28.3% (h=1) / 35.6% (h=4) -- alto en comparación con
cuántos hogares realmente cambian de **tamaño** entre `t` y `t+h`
(`n_miembros_t != n_miembros_th`, sin mirar sexo/edad de nadie). Si la brecha entre
ambos es grande y crece con el tamaño del hogar, es la misma señal mecánica de D10:
cada miembro tiene ~93% de chance de coincidir (D6), así que la chance de que **al
menos uno** de `n` falle sube con `n` aunque nadie se haya ido -- no necesariamente
cambio real.

In [ ]:
for h in HORIZONTES:
    df = pares_todos[h].copy()
    df["cambio_tamano"] = df["n_miembros_t"] != df["n_miembros_th"]
    df["tamano"] = df["n_miembros_t"].clip(upper=6)
    tabla = df.groupby("tamano").agg(
        n_pares=("tamano", "size"),
        pct_cambio_tamano=("cambio_tamano", lambda s: round(s.mean() * 100, 1)),
        pct_cambio_composicion=("cambio_composicion", lambda s: round(s.mean() * 100, 1)),
    )
    tabla["brecha_pp"] = (tabla["pct_cambio_composicion"] - tabla["pct_cambio_tamano"]).round(1)
    tabla.index = [str(i) if i < 6 else "6+" for i in tabla.index]
    print(f"--- h={h} ---")
    display(tabla)
    print(f"total: cambio_tamano={df['cambio_tamano'].mean():.1%}  cambio_composicion={df['cambio_composicion'].mean():.1%}")
    print()

**Diagnóstico, no implementado**: la brecha crece con el tamaño del hogar en ambos
horizontes (ver tabla) -- consistente con el mismo sesgo mecánico de D10 (a mayor `n`,
mayor chance de que al menos un miembro caiga fuera de la ventana de edad por azar o
error de declaración, no por un cambio real). Probé ensanchar la ventana de edad
(±2 en vez de ±1 para h=1, -1..+3 en vez de 0..+2 para h=4): baja el total a 22.3%/31.5%
pero la brecha con `cambio_tamano` sigue creciendo con `n` -- ensanchar la ventana
atenúa el problema, no lo resuelve.

**Dos opciones para redefinir el flag, ninguna implementada**:
- **Por cambio de cantidad de integrantes** (`n_miembros_t != n_miembros_th`): robusto al
  sesgo por tamaño (no depende de matchear personas), pero no detecta un reemplazo de
  integrante que mantiene el mismo tamaño (alguien se va, otra persona distinta entra).
- **Ventana de edad más amplia solo para este flag** (no para `identidad_nucleo`, que ya
  funciona bien con la ventana actual, D10): atenúa la brecha sin descartar el matcheo
  por persona, pero no la elimina -- el sesgo por tamaño persiste en menor escala.

Sin una recomendación cerrada todavía -- queda para decidir junto con la directora.

### Paso 3 — N real (tabla para el documento de la directora)

In [ ]:
tablas_n_real = {}
for h in HORIZONTES:
    df = pares_todos[h]
    if df.empty:
        tablas_n_real[h] = pd.DataFrame()
        continue
    tabla = df.groupby(["regimen_ingreso", "mandato_presidencial_t", "ANO4_t"]).agg(
        n_pares=("CODUSU", "size"),
        n_identidad_confirmada=("identidad_confirmada", lambda s: (s == True).sum()),  # noqa: E712
    ).reset_index()
    tablas_n_real[h] = tabla
    print(f"--- h={h}: por régimen de ingreso x mandato presidencial (agregado, sin desagregar por aglomerado en pantalla) ---")
    display(df.groupby(["regimen_ingreso", "mandato_presidencial_t"]).agg(
        n_pares=("CODUSU", "size"), n_identidad_confirmada=("identidad_confirmada", lambda s: (s == True).sum()),  # noqa: E712
    ))

In [ ]:
print("--- por aglomerado de origen (top 10 por cantidad de pares) ---")
for h in HORIZONTES:
    df = pares_todos[h]
    if df.empty:
        continue
    print(f"h={h}:")
    display(df.groupby("AGLOMERADO").size().sort_values(ascending=False).head(10).rename("n_pares"))

**Ejes de ingreso declarado y de ocupación** (NA pre-2016 contado aparte, no
mezclado con `False`):

In [ ]:
for h in HORIZONTES:
    df = pares_todos[h]
    if df.empty:
        continue
    print(f"--- h={h} ---")
    ambos_declarados = (df["ingreso_declarado_t"] == True) & (df["ingreso_declarado_th"] == True)
    algun_na = df["ingreso_declarado_t"].isna() | df["ingreso_declarado_th"].isna()
    print(f"pares con ingreso declarado en ambos extremos: {ambos_declarados.sum():,} "
          f"({ambos_declarados.sum()/len(df):.1%})")
    print(f"pares con algún extremo NA (histórico): {algun_na.sum():,} ({algun_na.sum()/len(df):.1%})")
    algun_ocupado = df["hay_ocupado_t"] | df["hay_ocupado_th"]
    print(f"pares con algún ocupado (en t o t+h): {algun_ocupado.sum():,} ({algun_ocupado.sum()/len(df):.1%})")
    print()

**Atrición descriptiva**: hogares con par vs. sin par en `t`, comparados por quintil
de `IPCF` (calculado dentro de cada trimestre, no pooled -- los pesos no son
comparables entre trimestres sin deflactar), `ESTADO` del jefe/a, cantidad de
miembros, y aglomerado. Solo descriptivo, sin conclusión:

In [ ]:
for h in HORIZONTES:
    partes = atricion_acumulada[h]
    if not partes:
        continue
    atricion = pd.concat(partes, ignore_index=True)
    atricion["quintil_ipcf"] = atricion.groupby(["anio_t", "trimestre_t"])["IPCF"].transform(
        lambda s: pd.qcut(s.rank(method="first"), 5, labels=False, duplicates="drop") + 1 if s.notna().sum() >= 5 else pd.NA
    )
    print(f"--- h={h}: estado de vinculación por quintil de IPCF en t ---")
    display((atricion.groupby("quintil_ipcf")["estado_vinculacion"].value_counts(normalize=True).unstack().round(3)))
    print(f"--- h={h}: estado de vinculación por ESTADO del jefe/a en t ---")
    display((atricion.groupby("estado_jefe")["estado_vinculacion"].value_counts(normalize=True).unstack().round(3)))
    print(f"--- h={h}: estado de vinculación por cantidad de miembros en t ---")
    display((atricion.groupby("n_miembros")["estado_vinculacion"].value_counts(normalize=True).unstack().round(3).head(10)))
    print()

## Escritura de agregados y registro de la corrida

In [ ]:
sufijo = "_parcial" if ES_CORRIDA_PARCIAL else ""

filas_globales = []
for h in HORIZONTES:
    df = pares_todos[h]
    if df.empty:
        continue
    tabla = df.groupby(["regimen_ingreso", "mandato_presidencial_t"]).agg(
        n_pares=("CODUSU", "size"),
        n_identidad_confirmada=("identidad_confirmada", lambda s: (s == True).sum()),  # noqa: E712
        n_identidad_no_confirmada=("identidad_confirmada", lambda s: (s == False).sum()),  # noqa: E712
        n_jefe_ambiguo=("identidad_confirmada", lambda s: s.isna().sum()),
        n_cambio_composicion=("cambio_composicion", "sum"),
    ).reset_index()
    tabla["h"] = h
    filas_globales.append(tabla)

destino_global = META / f"panel_resumen_pares{sufijo}.csv"
if filas_globales:
    pd.concat(filas_globales, ignore_index=True).to_csv(destino_global, index=False)
    print(f"guardado: {destino_global}" + (" -- CORRIDA PARCIAL, no pisa el global" if ES_CORRIDA_PARCIAL else ""))
else:
    print("sin pares para agregar -- no se escribió panel_resumen_pares.csv")

In [ ]:
n_filas_totales = sum(len(df) for df in pares_todos.values())
pm.registrar_corrida(
    META / "corridas.csv",
    notebook="02_panel_hogares",
    parametros={"TRIMESTRES": TRIMESTRES, "REHACER": REHACER, "HORIZONTES": HORIZONTES,
                "hash_panel": panel.hash_esquema_panel()},
    n_trimestres=len(trimestres_origen),
    n_filas=n_filas_totales,
    duracion_segundos=duracion_total,
    commit_git=COMMIT_GIT,
)

print("Pares h=1:", len(pares_todos.get(1, [])))
print("Pares h=4:", len(pares_todos.get(4, [])))
print("Pares escritos (nuevo):", n_escritos, "-- reusados (al día):", n_reusados)

## Resumen final consolidado, por horizonte y mandato presidencial

Una fila por horizonte (`mandato_presidencial_t = "TODOS"`) y una por cada mandato de
origen. `n_pares_vinculados`/`identidad_confirmada`/`cambio_composicion`/
`cambio_jefatura`/los tres ejes de N real son sobre los pares vinculados;
`hogar_reemplazado`/`vivienda_sin_par` son sobre el universo de viviendas de origen
(D9) -- distinto denominador, no se combinan en una sola tasa.

**D11**: se agregan las mismas métricas de composición/jefatura/ejes restringidas a la
subpoblación con `identidad_confirmada` (prefijo `confirmada_`), y
`n_ingreso_presente_historico_ambas_puntas` -- pares de la era histórica con `IPCF`
presente en ambas puntas (sin pasar por el flag `ingreso_declarado`, que ahí es `NA`
por no existir `PONDIH` antes de 2016, D3) -- para dimensionar si conviene usar `IPCF`
crudo en el eje de ingreso en vez de limitarlo a la era regular.

**Contexto, no eje (D15)**: esta tabla por mandato presidencial es descripción de
contexto, no organiza el análisis -- el eje del TFI es el riesgo de caer en mayor
informalidad o de no salir de ella, por quintil y ancla habitacional
(`docs/decisiones_metodologicas.md`, D15). Mismo criterio que el notebook 05.

In [ ]:
mandato_por_origen = {
    (anio, trimestre): panel.mandato_presidencial(anio, trimestre) for anio, trimestre in trimestres_origen
}
tasas_viviendas["mandato_presidencial_t"] = [
    mandato_por_origen.get((a, t)) for a, t in zip(tasas_viviendas["anio"], tasas_viviendas["trimestre"])
]

MANDATOS_EN_ORDEN = [nombre for _, nombre in panel.MANDATO_PRESIDENCIAL]

filas_resumen_final = []
for h in HORIZONTES:
    df = pares_todos[h]
    tv = tasas_viviendas[(tasas_viviendas["h"] == h) & tasas_viviendas["par_existe"]]

    for mandato in [None] + MANDATOS_EN_ORDEN:
        sub = df if mandato is None else df[df["mandato_presidencial_t"] == mandato]
        sub_tv = tv if mandato is None else tv[tv["mandato_presidencial_t"] == mandato]
        if sub.empty and sub_tv.empty:
            continue

        n_pares = len(sub)
        n_confirmada = int((sub["identidad_confirmada"] == True).sum())  # noqa: E712
        n_cambio_comp = int(sub["cambio_composicion"].sum())
        n_cambio_jef = int(sub["cambio_jefatura"].sum())

        ingreso_ambas_puntas_regular = (sub["regimen_ingreso"] == "regular") & sub["ingreso_declarado_t"] & sub["ingreso_declarado_th"]
        algun_ocupado = sub["hay_ocupado_t"] | sub["hay_ocupado_th"]
        n_ingreso_ambas = int(ingreso_ambas_puntas_regular.sum())
        n_algun_ocupado = int(algun_ocupado.sum())
        n_ambos_ejes = int((ingreso_ambas_puntas_regular & algun_ocupado).sum())

        # D11: mismas métricas de composición/jefatura/ejes, restringidas a los pares
        # con identidad confirmada -- subpoblación "limpia" para comparar contra el total.
        sub_conf = sub[sub["identidad_confirmada"] == True]  # noqa: E712
        n_conf = len(sub_conf)
        n_conf_cambio_comp = int(sub_conf["cambio_composicion"].sum())
        n_conf_cambio_jef = int(sub_conf["cambio_jefatura"].sum())
        conf_ingreso_ambas = (sub_conf["regimen_ingreso"] == "regular") & sub_conf["ingreso_declarado_t"] & sub_conf["ingreso_declarado_th"]
        conf_algun_ocupado = sub_conf["hay_ocupado_t"] | sub_conf["hay_ocupado_th"]
        n_conf_ingreso_ambas = int(conf_ingreso_ambas.sum())
        n_conf_algun_ocupado = int(conf_algun_ocupado.sum())
        n_conf_ambos_ejes = int((conf_ingreso_ambas & conf_algun_ocupado).sum())

        # D11: pares de la era histórica con IPCF presente en ambas puntas (sin pasar por
        # el flag ingreso_declarado, que en histórico es NA por no existir PONDIH antes de
        # 2016, D3) -- para dimensionar si conviene usar IPCF crudo en vez del flag ahí.
        historico = sub[sub["regimen_ingreso"] == "historico"]
        n_ingreso_presente_historico = int((historico["IPCF_t"].notna() & historico["IPCF_th"].notna()).sum())

        filas_resumen_final.append({
            "h": h,
            "mandato_presidencial_t": mandato if mandato is not None else "TODOS",
            "n_pares_vinculados": n_pares,
            "n_identidad_confirmada": n_confirmada,
            "pct_identidad_confirmada": round(n_confirmada / n_pares * 100, 1) if n_pares else None,
            "n_cambio_composicion": n_cambio_comp,
            "pct_cambio_composicion": round(n_cambio_comp / n_pares * 100, 1) if n_pares else None,
            "n_cambio_jefatura": n_cambio_jef,
            "pct_cambio_jefatura": round(n_cambio_jef / n_pares * 100, 1) if n_pares else None,
            "n_hogar_reemplazado": int(sub_tv["n_hogar_reemplazado"].sum()),
            "n_vivienda_sin_par": int(sub_tv["n_vivienda_sin_par"].sum()),
            "n_ingreso_declarado_ambas_puntas_regular": n_ingreso_ambas,
            "n_algun_ocupado": n_algun_ocupado,
            "n_ambos_ejes": n_ambos_ejes,
            "confirmada_n_pares": n_conf,
            "confirmada_n_cambio_composicion": n_conf_cambio_comp,
            "confirmada_pct_cambio_composicion": round(n_conf_cambio_comp / n_conf * 100, 1) if n_conf else None,
            "confirmada_n_cambio_jefatura": n_conf_cambio_jef,
            "confirmada_pct_cambio_jefatura": round(n_conf_cambio_jef / n_conf * 100, 1) if n_conf else None,
            "confirmada_n_ingreso_declarado_ambas_puntas_regular": n_conf_ingreso_ambas,
            "confirmada_n_algun_ocupado": n_conf_algun_ocupado,
            "confirmada_n_ambos_ejes": n_conf_ambos_ejes,
            "n_ingreso_presente_historico_ambas_puntas": n_ingreso_presente_historico,
        })

resumen_final_df = pd.DataFrame(filas_resumen_final)
sufijo = "_parcial" if ES_CORRIDA_PARCIAL else ""
destino_resumen_final = META / f"panel_resumen_final{sufijo}.csv"
resumen_final_df.to_csv(destino_resumen_final, index=False)
print(f"guardado: {destino_resumen_final} ({len(resumen_final_df)} filas)" +
      (" -- CORRIDA PARCIAL, no pisa el global" if ES_CORRIDA_PARCIAL else ""))
display(resumen_final_df)